# Real Data Performance
### Predict perfromance of real 3E cells using machine learning

In [1]:
#import libraries

# import pybamm 
import tensorflow as tf
#import gpflow
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt
import itertools
import os
import datetime

from scipy import integrate
import scipy.stats
import pickle
import mgzip
import pandas as pd
import importlib

from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
import math

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


# from statsmodels.graphics.tsaplots import plot_acf

import time
import keras_tuner as kt

from sklearn.preprocessing import RobustScaler

import subprocess
from subprocess import call
from xlsx2csv import Xlsx2csv


#### Please verify that the GPU is connected! It provides significant speed up for NN training.

In [2]:
from tensorflow.python.client import device_lib
def get_available_devices():
     local_device_protos = device_lib.list_local_devices()
     return [x.name for x in local_device_protos]
print(get_available_devices()) 

['/device:CPU:0', '/device:GPU:0']


In [3]:
#parameters

import datetime
date = datetime.date.today().strftime('%Y-%m-%d')

date = "2024-04-15"
dataset_filename = "data110323"


In [4]:
#reload data
xCC = pd.read_pickle(r'Features/' + date + r'/xCC.pkl')
yCC = pd.read_pickle(r'Features/' + date + r'/yCC.pkl')

In [5]:
with open(r'Features/' + date + r'/masterList.pkl', 'rb') as f:
    masterList = pickle.load(f)


totalCells = max(masterList.keys())
masterList

cell_num
1        0.33
2        0.33
3        0.33
4        0.33
5         0.5
6         0.5
7         0.5
8         0.5
9         0.5
10        0.8
11        0.8
12        0.8
13        0.8
14        0.8
15        0.8
16        0.8
17        0.8
18        0.8
19        0.8
20        0.8
21        0.8
22        0.8
23        0.8
24        0.8
25        0.8
26        0.8
27        0.8
28        0.8
29          1
30          1
31          1
32          1
33          1
34          1
35          1
36          1
37          1
38          1
39          1
40          1
41          1
42          1
43          1
44          1
45          1
46          1
47          1
48          1
49          1
50    stepped
51    stepped
52    stepped
53    stepped
54    stepped
55    stepped
Name: C-rate, dtype: object

In [6]:
#reload needed
from functions import TrainTestSplit
importlib.reload(TrainTestSplit)

#testing 4/19, 3.55 activation
# cell_test = [3,5, 9,12]

#testing 4/19, Canrd Li
# cell_test = [3,5,7]

#testing 4/19, all stepped in test
# cell_test = [11,12]

#0618-TotTime
#cell_test = [3,7,10,12,15,20,24,27,32,34]

#cell_test = [29,32,35,36,37,38,41,43,44,45,47,48]
cell_test = [3,8,15,19,30,31,34,51]

cell_train = np.array([x for x in masterList.keys() if x not in cell_test])

xTrain, yTrain, xTest, yTest = TrainTestSplit.train_test_split(xCC, yCC, 'cell', cell_test, masterList)
timeTrain = xTrain['Time']
timeTest = xTest['Time']

#5


print(xTrain.shape, xTest.shape)
print(len(xTrain)/(len(xCC)))

Updated
Training list:  [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55]
Testing list:  [3, 8, 15, 19, 30, 31, 34, 51]
(7523137, 65) (2162149, 65)
0.7767594059690132


## Feature Selection

In [7]:
input_scaler = RobustScaler()

xTrain_scaled = xTrain.loc[::8,:].copy()

xTrain_scaled.values[:]  = input_scaler.fit_transform(xTrain.loc[::8,:])



In [8]:
feature_names = xTrain_scaled.columns
print("All Features:")
print(feature_names)

All Features:
Index(['Voltage', 'Current', 'Throughput', 'Energy', 'Capacity', 'Time',
       'Overpotential', 'dQ_dV', 'dVoltage/dt', 'dEnergy/dt', 'dCapacity/dt',
       'dOverpotential/dt', 'Power', 'Voltage-8', 'Current-8', 'Throughput-8',
       'Energy-8', 'Capacity-8', 'Time-8', 'Overpotential-8', 'dQ_dV-8',
       'dVoltage/dt-8', 'dEnergy/dt-8', 'dCapacity/dt-8',
       'dOverpotential/dt-8', 'Power-8', 'Voltage-6', 'Current-6',
       'Throughput-6', 'Energy-6', 'Capacity-6', 'Time-6', 'Overpotential-6',
       'dQ_dV-6', 'dVoltage/dt-6', 'dEnergy/dt-6', 'dCapacity/dt-6',
       'dOverpotential/dt-6', 'Power-6', 'Voltage-4', 'Current-4',
       'Throughput-4', 'Energy-4', 'Capacity-4', 'Time-4', 'Overpotential-4',
       'dQ_dV-4', 'dVoltage/dt-4', 'dEnergy/dt-4', 'dCapacity/dt-4',
       'dOverpotential/dt-4', 'Power-4', 'Voltage-2', 'Current-2',
       'Throughput-2', 'Energy-2', 'Capacity-2', 'Time-2', 'Overpotential-2',
       'dQ_dV-2', 'dVoltage/dt-2', 'dEnergy/dt-2', '

In [8]:
# from sklearn.svm import SVR
from sklearn.feature_selection import RFE

# Create the RFE object and rank each pixel
rfr = RandomForestRegressor(n_estimators=30, n_jobs = 32)
rfe = RFE(estimator=rfr, n_features_to_select=5, step=15,verbose = 2)
rfe.fit(xTrain_scaled, yTrain.loc[::8])
ranking = rfe.ranking_


Fitting estimator with 65 features.
Fitting estimator with 50 features.
Fitting estimator with 35 features.
Fitting estimator with 20 features.


In [9]:
#names of selected features
xTrain.iloc[:, rfe.ranking_ == 1].columns

Index(['Voltage', 'Energy', 'Capacity', 'Energy-8', 'Voltage-2'], dtype='object')

In [10]:
#obtain feature importance values
forest_regr = RandomForestRegressor(n_estimators=30, n_jobs=32, verbose =2 )
forest_regr.fit( xTrain_scaled.iloc[:,rfe.ranking_ == 1], yTrain.loc[::8]) 
print(forest_regr.feature_importances_)
xTrain.loc[:,rfe.ranking_ == 1].columns

[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.


building tree 1 of 30building tree 2 of 30
building tree 3 of 30

building tree 4 of 30
building tree 5 of 30
building tree 6 of 30
building tree 7 of 30
building tree 8 of 30
building tree 9 of 30
building tree 10 of 30
building tree 11 of 30
building tree 12 of 30
building tree 13 of 30
building tree 14 of 30
building tree 15 of 30
building tree 16 of 30
building tree 17 of 30
building tree 18 of 30
building tree 19 of 30
building tree 20 of 30
building tree 21 of 30
building tree 22 of 30
building tree 23 of 30
building tree 24 of 30
building tree 25 of 30
building tree 26 of 30
building tree 27 of 30
building tree 28 of 30
building tree 29 of 30
building tree 30 of 30


[Parallel(n_jobs=32)]: Done  15 out of  30 | elapsed:   10.5s remaining:   10.5s


[0.12673255 0.58842017 0.0977404  0.12845544 0.05865145]


[Parallel(n_jobs=32)]: Done  30 out of  30 | elapsed:   13.4s finished


Index(['Voltage', 'Energy', 'Capacity', 'Energy-8', 'Voltage-2'], dtype='object')

#### Now that we know the important features, we can drop columns to obtain only the useful ones.

In [7]:
#Start of the reloading process

#Activate this line on the first model building
try:
    use_cols = xTrain.iloc[:, rfe.ranking_ == 1].columns
except:
    print('Using specified columns')
    use_cols = ['Energy', 'Capacity', 'Voltage-8', 'Energy-8', 'Power-8']
    #use_cols = ['Voltage', 'Current']

xTrain = xTrain[use_cols]
xTest = xTest[use_cols]

Using specified columns


## Model Development

In [8]:
#Scale the data. Required for GPR
input_scaler = RobustScaler()


xTrain_scaled = xTrain.copy()
xTrain_minmax = input_scaler.fit_transform(xTrain)
xTrain_scaled.values[:]  = xTrain_minmax

#with open(r'Features/' + date + r'/scaler.pkl', 'wb') as f:
    #pickle.dump(input_scaler, f)

xTest_minmax = input_scaler.transform(xTest)

xTrain_minmax.shape



#xTrain_minmax = xTrain
#xTest_minmax = xTest

(7523137, 5)

In [13]:
if not os.path.exists('models/' + date + '/' + dataset_filename):
    os.makedirs('models/' + date + '/' + dataset_filename)

In [9]:
#Make sure to scale first
#Need to run in reloading process

tf.random.set_seed(1)
from tensorflow.keras import layers
# from tensorflow.python.keras.models import Sequential
from tensorflow.keras.models import Sequential
# from tensorflow.python.keras.wrappers.scikit_learn import KerasRegressor
import keras

#Scale the data. Required for GPR

from sklearn.pipeline import make_pipeline


from sklearn.model_selection import train_test_split
#pipe = make_pipeline(MinMaxScaler(), GaussianProcessRegressor())



learning_rate = 0.001
training_steps = 10000
batch_size = 128
display_step = 200

# Network Parameters
# num_input =  # 
timesteps = 30 # timesteps to use
step = 1 #skip data (seconds) to save on time

num_hidden = 128 # hidden layer num of features
input_dim = xTrain.shape[1] # number of features

xTrain_temp = xTrain.iloc[::step, :].copy()
yTrain_temp = yTrain.iloc[::step].copy()

input_scaler = RobustScaler()

#with open(r"Features/"  + "2023-07-09/" + "scaler2.pkl", 'rb') as f:
    #input_scaler = pickle.load(f)


#xTrain_minmax = input_scaler.transform(xTrain.iloc[::step, :])

xTrain_minmax = input_scaler.fit_transform(xTrain.iloc[::step, :])

with open(r'Features/' + date + r'/scaler.pkl', 'wb') as f:
    pickle.dump(input_scaler, f)

xTest_minmax = input_scaler.transform(xTest)

yExp = yTest.to_numpy().flatten()

timeTest = timeTest.copy()

xTrain_temp.iloc[:,:] = xTrain_minmax

In [11]:
def model_builder():


    model = Sequential()

    #change hidden units

    model.add(layers.Dense(5, input_dim = len(xTrain.columns), kernel_initializer = 'normal', bias_initializer= tf.keras.initializers.HeNormal,  activation = tf.nn.relu))
    model.add(layers.Dense(units = 256, activation = tf.nn.relu, bias_initializer=tf.keras.initializers.HeNormal, kernel_initializer = 'normal'))
    model.add(layers.Dense(units = 256, activation = tf.nn.leaky_relu, bias_initializer=tf.keras.initializers.HeNormal, kernel_initializer = 'normal'))

    # if(num_models == 1):
    #     model.add(layers.Dense(48,activation = tf.nn.leaky_relu, bias_initializer=tf.keras.initializers.HeNormal, dtype=tf.float32))
    
    model.add(layers.Dense(1, activation = 'linear', dtype=tf.float32))

    optimizer = tf.keras.optimizers.legacy.SGD(learning_rate=learning_rate, momentum=0.001)
    model.compile(loss = 'mse', optimizer = optimizer, metrics = ['mse', 'mape'])

    return model

class MyHyperModel(kt.HyperModel):
    def build(self, hp):
        model = keras.Sequential()
        hp_units_1 = hp.Int('units', min_value = 32, max_value = 512, step = 32)
        hp_units_2 = hp.Int('units', min_value = 32, max_value = 512, step = 32)

        model.add(layers.Dense(5, input_dim = len(xTrain.columns), kernel_initializer = 'normal', bias_initializer= tf.keras.initializers.HeNormal,  activation = tf.nn.relu))
        model.add(layers.Dense(units = hp_units_1, activation = tf.nn.relu, bias_initializer=tf.keras.initializers.HeNormal, kernel_initializer = 'normal'))
        model.add(layers.Dense(units = hp_units_2, activation = tf.nn.leaky_relu, bias_initializer=tf.keras.initializers.HeNormal, kernel_initializer = 'normal'))
        model.add(layers.Dense(1, activation = 'linear', dtype=tf.float32))

        optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
        model.compile(loss = 'mse', optimizer = optimizer, metrics = ['mse', 'mape'])


        return model

    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            *args,
            batch_size=hp.Choice("batch_size", [64, 128, 256]),
            **kwargs,
        )
    



#### Ensemble Model

In [12]:
nn_models = []
nn_history = []
yPred_list = []


#performance not very good with c_rate based split
optimizer = tf.keras.optimizers.legacy.SGD(learning_rate=learning_rate, momentum=0.001)

num_models = 5
#create 5 models
with tf.device('/gpu:0'):
    for i in range(num_models):

        #i = 4
        rows = np.random.choice(len(xTrain_temp), size = int(len(xTrain_temp)*0.7))
        #print(f"\n{rows}")

        

        xTrain_sample = xTrain_temp.iloc[rows,:]
        yTrain_sample = yTrain_temp.iloc[rows]
        # break

        model = model_builder()
        
        nn_models.append(model)
        nn_history.append(model.fit(xTrain_sample, yTrain_sample, epochs=70, batch_size = 256, verbose = 1, shuffle=True))

        if(num_models != 1):
            model.save(filepath='models/' + date + '/' + dataset_filename + '/ensemble/model_' + str(i))

        yPred_list.append(model.predict(xTest_minmax).flatten())
        
    yPred = np.array(yPred_list).mean(axis = 0)

    if(num_models == 1):
        
        yPred_baseline = yPred
        model.save(filepath='models/' + date + '/' + dataset_filename + '/baseline')
    else:
        yPred_ensemble = yPred
        # for i in range(num_models):
        
        
        
    print(model.summary())


Epoch 1/70
18751/18751 [==============================] - 22s 1ms/step - loss: 0.0026 - mse: 0.0026 - mape: 428.0184
Epoch 2/70
18751/18751 [==============================] - 21s 1ms/step - loss: 0.0014 - mse: 0.0014 - mape: 254.9381
Epoch 3/70
18751/18751 [==============================] - 21s 1ms/step - loss: 0.0014 - mse: 0.0014 - mape: 247.4808
Epoch 4/70
18751/18751 [==============================] - 21s 1ms/step - loss: 0.0014 - mse: 0.0014 - mape: 244.6400
Epoch 5/70
18751/18751 [==============================] - 20s 1ms/step - loss: 0.0013 - mse: 0.0013 - mape: 239.9654
Epoch 6/70
18751/18751 [==============================] - 21s 1ms/step - loss: 0.0013 - mse: 0.0013 - mape: 235.7591
Epoch 7/70
18751/18751 [==============================] - 21s 1ms/step - loss: 0.0013 - mse: 0.0013 - mape: 228.9714
Epoch 8/70
18751/18751 [==============================] - 20s 1ms/step - loss: 0.0012 - mse: 0.0012 - mape: 222.1434
Epoch 9/70
18751/18751 [==============================] - 20s 1m

: 

## Reload gp model

In [10]:
try:
    with open('models/' + date + '/' + dataset_filename + '/svgp_model.pkl', 'rb') as f:
        params = pickle.load(f)

    gpflow.utilities.multiple_assign(m, params)
except Exception as e:
    print(e)
yPred_gp = np.loadtxt('models/' + date + '/' + dataset_filename + 'svgp_results.npy')

No module named 'gpflow'


### Reload Ensemble Models

In [11]:
#reload NN models

nn_models = []
yPred_list = []
# dateCopy = date.copy()

dateCopy ="2024-04-15" 

# model = tf.keras.models.load_model('models/' + date + '/' + dataset_filename + '/baseline')
# yPred_list.append(model.predict(xTest_minmax).flatten())
# yPred = np.array(yPred_list).mean(axis = 0)
# yPred_baseline = yPred
#num_models incicates the number of models you have
num_models = 5
#create 5 models
yPred_list = []
for i in range(num_models):
    
    model = tf.keras.models.load_model('models/' + dateCopy + '/' + dataset_filename + '/ensemble/model_' + str(i))
    yPred_list.append(model.predict(xTest_minmax).flatten())
    
yPred = np.array(yPred_list).mean(axis = 0)
yPred_ensemble = yPred
    

67568/67568 [==============================] - 32s 467us/step


#### Baseline Linear Regression Model

In [14]:
from sklearn.linear_model import LinearRegression

baseline = LinearRegression()
reg = baseline.fit(xTrain_temp, yTrain)
print(reg.coef_)
yPred_baseline = reg.predict(xTest_minmax)
# pickle.dumps(baseline)



filename = open('models/' + date + '/' + dataset_filename + '/baseline.pkl', 'wb')
pickle.dump(reg, filename)
filename.close()

[-0.11892341  0.27463639 -0.02064804 -0.23788703 -0.01694865]


C:\Users\Li group ML\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:409: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## Evaluate Model Performance

In [19]:
##linear regression for observing relationship between predicted and actual data

slope, intercept, r, p, stderr = scipy.stats.linregress(yPred.T, yExp.T)
line = f'Regression line: y={intercept:.2f}+{slope:.2f}x, r={r:.2f}'

plt.figure()
plt.plot(yPred, yExp, '*', label='data', markersize = 0.1)
plt.plot(yPred, intercept + slope * yPred, label=line)

plt.legend()
plt.xlabel('Predicted NEP')
plt.ylabel('Actual NEP')
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(xlabel='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 15)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 15)

In [13]:

#specify which model to evaluate metrics for
yPred = yPred_ensemble

In [32]:
from sklearn.metrics import mean_squared_error

#Get error metrics
#* 100 / len (yPred)
mpe = abs(yPred - yExp)
mae = mpe.mean()

rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100
percent_rmse = (rmse/np.mean(yExp))*100
percent_rmse2 = (rmse/(max(yExp)-min(yExp)))*100

print('MAE:', "{:.5f}".format(mae), 'V')
# print('RMSPE:', "{:.5f}".format
# (rmspe), '%')
print('RMSE:', "{:.5f}".format(rmse), 'V')
print('RMSE%:', "{:.5f}".format(percent_rmse), '%')
print('RMSPE%:', "{:.5f}".format(rmspe), 'V')
##plot absolute differences


plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)

# plt.legend()
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(x  label='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

MAE: 0.01710 V
RMSE: 0.02277 V
RMSE%: 29.95573 %
RMSPE%: 10231.81796 V


In [12]:
#Calculate the overall RMSE, cell-to-cell standard deviation
from sklearn.metrics import mean_squared_error
yPred = yPred_ensemble
#Get error metrics
#* 100 / len (yPred)
mpe = abs(yPred - yExp)
mae = mpe.mean()

overall_rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100

print('MAE:', "{:.5f}".format(mae), 'V')
# print('RMSPE:', "{:.5f}".format
# (rmspe), '%')
print('RMSE:', "{:.5f}".format(overall_rmse), 'V')
##plot absolute differences


plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)

# plt.legend()
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(x  label='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

print('Averages across cells')

mae_list = []
rmse_list = []
numerator = 0
denominator = 0

for cell_num in cell_test:
    idx = [xTest.index.get_level_values(0) == cell_num][0]
    mae = abs(yPred[idx] - yExp[idx]).mean()
    rmse = mean_squared_error(yExp[idx], yPred[idx], squared=False)
    print('Dataset size: ', len(yPred[idx]) )
    print('MAE:', "{:.5f}".format(mae), 'V')
    print('RMSE:', "{:.5f}".format(rmse), 'V')
    numerator += len(yPred[idx]) * ((rmse - overall_rmse)**2)
    denominator += len(yPred[idx])
weighted_standard_deviation = (numerator/denominator) ** 0.5
print('standard deviation:', "{:.5f}".format(weighted_standard_deviation), 'V')

print(min(yExp),max(yExp))


MAE: 0.01710 V
RMSE: 0.02277 V
Averages across cells
Dataset size:  814100
MAE: 0.01382 V
RMSE: 0.01576 V
Dataset size:  557242
MAE: 0.01155 V
RMSE: 0.01540 V
Dataset size:  94132
MAE: 0.01983 V
RMSE: 0.02736 V
Dataset size:  167087
MAE: 0.03507 V
RMSE: 0.03864 V
Dataset size:  193011
MAE: 0.02103 V
RMSE: 0.02728 V
Dataset size:  76784
MAE: 0.03161 V
RMSE: 0.04026 V
Dataset size:  90403
MAE: 0.02004 V
RMSE: 0.02930 V
Dataset size:  169390
MAE: 0.01931 V
RMSE: 0.02676 V
standard deviation: 0.00828 V
-0.12474004000000001 0.4187494400000001


In [59]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Generate some random data
np.random.seed(0)
data_x = np.random.randn(100)
data_y = np.random.randn(100)
plt.figure(figsize=(15,13))

# Plot the KDE plot as a heatmap
ax = sns.kdeplot(x=data_x, y=data_y, shade=True, cmap="Blues")

# Adjust the z-order of ticks and axes
ax.set_zorder(1)  # Set the z-order of the KDE plot
plt.tick_params(axis='both', which='both', zorder=2)  # Set the z-order of ticks and axes
ax = plt.gca()
# Set border color and width for all four sides
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(3)

# Set ticks properties
ax.tick_params(axis='both', direction='in', which='major', labelsize=20, length=8, width=5, colors='black', zorder = 10)

# Set ticks position
ax.xaxis.set_tick_params(width=3, which='both', direction='in', bottom=True, top=False)
ax.yaxis.set_tick_params(width=3, which='both', direction='in', left=True, right=False)
# Adjust the z-order of ticks and spines
plt.gca().spines['left'].set_zorder(10)
plt.gca().spines['bottom'].set_zorder(10)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.xlabel('Measured Anode Voltage', fontsize = 25, weight = 'bold')
plt.ylabel('Predicted Anode Voltage', fontsize = 25, weight = 'bold')
plt.tick_params(axis='both', which='both', zorder=10)
plt.tight_layout()


# Show plot
plt.show()

C:\Users\Li group ML\AppData\Local\Temp\ipykernel_15008\3829964782.py:12: FutureWarning: 

`shade` is now deprecated in favor of `fill`; setting `fill=True`.
This will become an error in seaborn v0.14.0; please update your code.

  ax = sns.kdeplot(x=data_x, y=data_y, shade=True, cmap="Blues")


In [22]:
#Calculate the R2 instead of pearson relationship coefficient

def calculate_r_squared(y_true, y_pred):
    """
    Calculate R-squared (coefficient of determination) for regression.
    
    Parameters:
    y_true : array-like, true values
    y_pred : array-like, predicted values
    
    Returns:
    r_squared : float, R-squared value
    """
    # Calculate the mean of the true values
    mean_true = np.mean(y_true)
    
    # Calculate the total sum of squares (TSS)
    total_sum_squares = np.sum((y_true - mean_true)**2)
    
    # Calculate the residual sum of squares (RSS)
    residual_sum_squares = np.sum((y_true - y_pred)**2)
    
    # Calculate R-squared
    r_squared = 1 - (residual_sum_squares / total_sum_squares)
    
    return r_squared

#filter the R2 for specific anode voltage domain:

r_squared_NN = calculate_r_squared(yExp, (yPred_ensemble))
r_squared_BL = calculate_r_squared(yExp, yPred_baseline)
r_squared_GP = calculate_r_squared(yExp, yPred_gp)

print("R-squared-ensemble:", r_squared_NN)
print("R-squared-baseline:", r_squared_BL)
print("R-squared-GP:", r_squared_GP)

R-squared-ensemble: 0.8480599661197401
R-squared-baseline: 0.7082118198498332
R-squared-GP: 0.8424830992843947


In [16]:
##linear regression comparison for all cells(2D density plots)
from scipy.stats import pearsonr
import matplotlib.patches as patches
#  'gp': yPred_gp
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['font.weight'] = 'bold'
# yPred_list = {'linear regression': yPred_baseline, 'ensemble': yPred_ensemble,'gp': yPred_gp}
# color_list = {'linear regression': 'Reds', 'ensemble': 'Greens','gp': 'Purples'}
yPred_list = {'ensemble': yPred_ensemble}
color_list = {'ensemble': 'Greens',}
# cycle = np.arange(0, 10, 1)
import seaborn as sns
matplotlib.rcParams['font.family'] = 'Arial'

#decide which cbar ticks you want to specify
#cbar_kws = {"ticks":np.linspace(0.0, 300.0, num = 11)}
#cbar_kws = {"label": "Kernel Density", "ticks": []}
cbar_kws = {"ticks": []}

#indList = timeTest.index.get_loc(tuple([cell_test[select_cell]]))
plt.figure(figsize=(12,12))

ax = plt.gca()
# Set border color and width for all four sides
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(3)

# Set ticks properties
ax.tick_params(axis='both', direction='in', which='major', labelsize=20, length=8, width=5, colors='black', zorder = 10)

# Set ticks position
ax.xaxis.set_tick_params(width=3, which='both', direction='in', bottom=True, top=False)
ax.yaxis.set_tick_params(width=3, which='both', direction='in', left=True, right=False)
plt.tight_layout()

corr, _ = pearsonr(yExp, yPred_list['ensemble'])
corr *=100
df = pd.DataFrame({'Measured Anode Voltage': yExp, 'Predicted Anode Voltage': yPred_list['ensemble']})
sns.kdeplot(data = df, x = 'Measured Anode Voltage', y = 'Predicted Anode Voltage', cmap = "rocket_r", fill = True, thresh = 0, levels = 100, cbar = False)
#sns.kdeplot(data = df, x = 'Measured Anode Voltage', y = 'Predicted Anode Voltage', cmap = "rocket_r", fill = True, thresh = 0, levels = 100, cbar = True,cbar_kws = cbar_kws)
#cbar_ax = cax.figure.axes[-1]
#cbar_ax.set_ticks(["Low","Medium",'High'])
#cbar_ax.tick_params(direction = 'in', labelsize = 25)
#cbar_ax.set_ylabel('Kernel Density', fontsize = 30)

plt.plot([-0.06,0.25],[-0.06,0.25], 'k',linewidth = 4)

plt.xlabel('Measured Anode Voltage', fontsize = 25, weight = 'bold')
plt.ylabel('Predicted Anode Voltage', fontsize = 25, weight = 'bold')

plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9)

# Show plot
plt.show()

In [24]:
#error of individual cycles

plt.figure()

yPred_list = {'linear regression': yPred_baseline, 'ensemble': yPred_ensemble,'gp': yPred_gp}
#yPred_list = {'gp': yPred_gp}

#  'ensemble': yPred_ensemble,'gp': yPred_gp

cycle_error = {'linear regression': [], 'ensemble': [],'gp': []}
#cycle_error = {'gp': []}

# ,'gp': []
# yPred_list = {'baseline': yPred_baseline}
# cycle_error = {'baseline': []}
select_cell = 4

maxCycles = xTest.loc[cell_test[select_cell]].index.max()

for key, value in yPred_list.items():
    list_keep = []
    for cycle in range(1, maxCycles+1, 1):
        try:
            indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle])) 
            cycle_error[key].append(mean_squared_error(yExp[indList], value[indList], squared=False))
            list_keep.append(cycle-1)
        except:
            # print(key)
            cycle_error[key].append(np.nan)
      
    cycles = np.arange(1, maxCycles+1, 1)
    plt.plot([cycles[i] for i in list_keep],[cycle_error[key][i] for i in list_keep], '*-', linewidth = 2, label = key)
    plt.ylim(0, 0.1)


ax = plt.gca()
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins = 10, integer=True))
plt.xlabel('Cycle')
plt.ylabel('RMSE [V]')
plt.legend(prop = {'size':10})
plt.savefig('models/' + date + '/' + dataset_filename + '/individual_cycle_error_' + str(select_cell))
print(cycle_error['ensemble'][98])

0.042959523375014345


In [39]:
#Put all plots together
import matplotlib.patches as patches
cycle = 20
plt.rcParams['axes.labelsize'] = 28
plt.rcParams['font.weight'] = 'bold'
matplotlib.rcParams['font.family'] = 'Arial'
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('white')


# cell_num = cell_test
select_cell = 1
indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle]))
indList1 = timeTest.index.get_loc(tuple([cell_test[select_cell], 40]))
indList2 = timeTest.index.get_loc(tuple([cell_test[select_cell], 60]))
indList3 = timeTest.index.get_loc(tuple([cell_test[select_cell], 100]))


axs[0].plot(timeTest.iloc[indList]*1.59/3600, yPred_ensemble[indList], linewidth = 4, linestyle = 'dashed', label = 'Predicted-Cycle 20', color = '#2166ac')
axs[0].plot(timeTest.iloc[indList]*1.59/3600, yExp[indList], linewidth = 4, label = 'Measured-Cycle 20', color = '#2166ac')

axs[0].plot(timeTest.iloc[indList1]*1.59/3600, yPred_ensemble[indList1], linewidth = 4, linestyle = 'dashed', label = 'Predicted-Cycle 40', color = '#4393c3')
axs[0].plot(timeTest.iloc[indList1]*1.59/3600, yExp[indList1], linewidth = 4, label = 'Measured-Cycle 40', color = '#4393c3')

axs[0].plot(timeTest.iloc[indList2]*1.59/3600, yPred_ensemble[indList2], linewidth = 4, linestyle = 'dashed', label = 'Predicted-Cycle 60', color = '#92c5de')
axs[0].plot(timeTest.iloc[indList2]*1.59/3600, yExp[indList2], linewidth = 4, label = 'Measured-Cycle 60', color = '#92c5de')

axs[0].plot(timeTest.iloc[indList3]*1.59/3600, yPred_ensemble[indList3], linewidth = 4, linestyle = 'dashed', label = 'Predicted-Cycle 100', color = '#d1e5f0')
axs[0].plot(timeTest.iloc[indList3]*1.59/3600, yExp[indList3], linewidth = 4, label = 'Measured-Cycle 100', color = '#d1e5f0')

# axs[0].set(xlabel =  'Times [s]', ylabel = 'Anode Potential [V]', yticks = [0, 0.1, 0.2, 0.3, 0.4, 0.5] )
axs[0].set(xlabel =  'Capacity (mAh)', ylabel = 'Anode Voltage (V)', xticks = [0,1,2,3], yticks = [-0.1, -0.05, 0, 0.05, 0.1, 0.15, 0.2])
axs[0].set_xlabel('Capacity (mAh)', weight = 'bold')
axs[0].set_ylabel('Anode Voltage (V)', weight = 'bold')
axs[0].legend(prop={'size':18})
ax = axs[0]
# Set border color and width for all four sides
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(3)

# Set ticks properties
ax.tick_params(axis='both', direction='in', which='major', labelsize=28, length=8, width=5, colors='black', zorder = 10)

# Set ticks position
ax.xaxis.set_tick_params(width=3, which='both', direction='out', bottom=True, top=False)
ax.yaxis.set_tick_params(width=3, which='both', direction='out', left=True, right=False)
plt.tight_layout()
for axis in ['bottom','top','left','right']:
    ax.spines[axis].set_linewidth(3)
ax.tick_params(axis = 'both', direction = 'in', which = 'major', labelsize = 28, length = 10, width = 5)
# r1 = patches.Rectangle((0, -0.11), 3, 0.11, color = "grey", alpha = 0.5)
# ax.add_patch(r1)
#axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_baseline[indList]), linewidth = 1, label = 'baseline')


axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_ensemble[indList]), linewidth = 1, label = 'ensemble')

#axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_gp[indList]), linewidth = 1, label = 'SVGP')

axs[1].set(xlabel =  'Times (s)', ylabel = 'Absolute V Difference'  )



#fig.suptitle('Anode Potential Predictions for Cycle ' + str(cycle) ) 

#print(abs(yPred_baseline - yExp).mean())
print(abs(yPred_ensemble - yExp).mean())
# print(abs(yPred_gp - yExp).mean())

plt.savefig('models/' + date + '/' + dataset_filename + '/one_cycle_error_comparison__cell_' + str(select_cell) + '_cycle_' + str(cycle))


0.017103600975670766


In [28]:
import matplotlib.patches as patches
cycle = 100
plt.rcParams['axes.labelsize'] = 28
plt.rcParams['font.weight'] = 'bold'
matplotlib.rcParams['font.family'] = 'Arial'
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('white')


# cell_num = cell_test
select_cell = 1
indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle]))
#x_predict =  timeTest.iloc[indList]*1.59/3600
#y_predict =  yPred_ensemble[indList]
# indList_lstm = timeTest.index.get_loc(tuple([cell_test, i]))
#axs[0].plot(timeTest.iloc[indList], yPred_baseline[indList], linewidth = 2, label = 'linear regression, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_baseline, squared=False)))
#axs[0].plot(timeTest.iloc[indList], yPred_ensemble[indList], linewidth = 2, label = 'ensemble, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_ensemble, squared=False)))
axs[0].plot(timeTest.iloc[indList]*1.59/3600, yPred_ensemble[indList], linewidth = 4, label = 'Predicted Anode Voltage', color = 'b')
#axs[0].plot(timeTest.iloc[indList], yPred_gp[indList], linewidth = 2, label = 'SVGP, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_gp, squared=False)))
axs[0].plot(timeTest.iloc[indList]*1.59/3600, yExp[indList], linewidth = 4, label = 'Measured Anode Voltage', color = 'r')

# axs[0].set(xlabel =  'Times [s]', ylabel = 'Anode Potential [V]', yticks = [0, 0.1, 0.2, 0.3, 0.4, 0.5] )
axs[0].set(xlabel =  'Capacity (mAh)', ylabel = 'Anode Voltage (V)', xticks = [0,1,2,3], yticks = [-0.1, -0.05, 0, 0.05, 0.1, 0.15, 0.2])
axs[0].set_xlabel('Capacity (mAh)', weight = 'bold')
axs[0].set_ylabel('Anode Voltage (V)', weight = 'bold')
axs[0].legend(prop={'size':18})
ax = axs[0]
# Set border color and width for all four sides
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(3)

# Set ticks properties
ax.tick_params(axis='both', direction='in', which='major', labelsize=28, length=8, width=5, colors='black', zorder = 10)

# Set ticks position
ax.xaxis.set_tick_params(width=3, which='both', direction='out', bottom=True, top=False)
ax.yaxis.set_tick_params(width=3, which='both', direction='out', left=True, right=False)
plt.tight_layout()
for axis in ['bottom','top','left','right']:
    ax.spines[axis].set_linewidth(3)
ax.tick_params(axis = 'both', direction = 'in', which = 'major', labelsize = 28, length = 10, width = 5)
# r1 = patches.Rectangle((0, -0.11), 3, 0.11, color = "grey", alpha = 0.5)
# ax.add_patch(r1)
#axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_baseline[indList]), linewidth = 1, label = 'baseline')


axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_ensemble[indList]), linewidth = 1, label = 'ensemble')

#axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_gp[indList]), linewidth = 1, label = 'SVGP')

axs[1].set(xlabel =  'Times (s)', ylabel = 'Absolute V Difference'  )



#fig.suptitle('Anode Potential Predictions for Cycle ' + str(cycle) ) 

#print(abs(yPred_baseline - yExp).mean())
print(abs(yPred_ensemble - yExp).mean())
# print(abs(yPred_gp - yExp).mean())

plt.savefig('models/' + date + '/' + dataset_filename + '/one_cycle_error_comparison__cell_' + str(select_cell) + '_cycle_' + str(cycle))


0.017103600975670766
